# Breast Cancer Classification

**Dataset:** Breast Cancer Wisconsin (Diagnostic) – [Kaggle](https://www.kaggle.com/datasets/uciml/breast-cancer-wisconsin-data)

**Goal:** Classify tumours as **Malignant (M)** or **Benign (B)** using four classifiers:
- Logistic Regression
- K-Nearest Neighbours (KNN)
- Random Forest
- Support Vector Machine (SVM)


## 1. Setup

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.neighbors import KNeighborsClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.svm import SVC
from sklearn.metrics import (
    accuracy_score, classification_report,
    confusion_matrix, ConfusionMatrixDisplay
)

RANDOM_STATE = 42
np.random.seed(RANDOM_STATE)

## 2. Exploratory Data Analysis

In [ ]:
# Load the dataset (update path if running locally)
df = pd.read_csv('data.csv')

print('Shape:', df.shape)
df.head()

In [ ]:
# Data types and a quick overview
df.info()

In [ ]:
# Check for missing values
print('Missing values per column:')
print(df.isnull().sum()[df.isnull().sum() > 0])

In [ ]:
# Drop unnecessary columns:
#   'id'          - just a row identifier, no predictive value
#   'Unnamed: 32' - entirely NaN (empty trailing column in the CSV)
df.drop(columns=['id', 'Unnamed: 32'], inplace=True)

print('Shape after dropping columns:', df.shape)
df.head()

In [ ]:
# Countplot of the diagnosis column using the 'magma' palette
fig, ax = plt.subplots(figsize=(6, 4))
sns.countplot(
    data=df,
    x='diagnosis',
    palette='magma',
    order=['B', 'M'],
    ax=ax
)
ax.set_title('Diagnosis Distribution', fontsize=14)
ax.set_xlabel('Diagnosis  (B = Benign, M = Malignant)')
ax.set_ylabel('Count')

# Annotate bars with counts
for p in ax.patches:
    ax.annotate(f'{int(p.get_height())}',
                (p.get_x() + p.get_width() / 2, p.get_height()),
                ha='center', va='bottom', fontsize=12)

plt.tight_layout()
plt.show()

In [ ]:
# Scatter plot: radius_mean vs texture_mean coloured by diagnosis
palette = {'B': '#fdae61', 'M': '#d7191c'}
fig, ax = plt.subplots(figsize=(7, 5))
for label, grp in df.groupby('diagnosis'):
    ax.scatter(grp['radius_mean'], grp['texture_mean'],
               label=label, alpha=0.6, color=palette[label], edgecolors='white', linewidths=0.3)
ax.set_xlabel('Radius Mean')
ax.set_ylabel('Texture Mean')
ax.set_title('Radius Mean vs Texture Mean by Diagnosis')
ax.legend(title='Diagnosis')
plt.tight_layout()
plt.show()

In [ ]:
# Scatter plot: concavity_mean vs concave points_mean
fig, ax = plt.subplots(figsize=(7, 5))
for label, grp in df.groupby('diagnosis'):
    ax.scatter(grp['concavity_mean'], grp['concave points_mean'],
               label=label, alpha=0.6, color=palette[label], edgecolors='white', linewidths=0.3)
ax.set_xlabel('Concavity Mean')
ax.set_ylabel('Concave Points Mean')
ax.set_title('Concavity Mean vs Concave Points Mean by Diagnosis')
ax.legend(title='Diagnosis')
plt.tight_layout()
plt.show()

## 3. Data Preprocessing

In [ ]:
# Counts of unique values in the diagnosis column
print('Unique value counts in diagnosis:')
print(df['diagnosis'].value_counts())

In [ ]:
# Map categorical labels to numeric:
#   M (Malignant) -> 1
#   B (Benign)    -> 0
df['diagnosis'] = df['diagnosis'].map({'M': 1, 'B': 0})

print('After mapping:')
print(df['diagnosis'].value_counts())
df.head()

In [ ]:
# Feature matrix and target vector
X = df.drop(columns=['diagnosis'])
y = df['diagnosis']

# Train / test split (80 / 20) with stratification
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=RANDOM_STATE, stratify=y
)

print('Training set:', X_train.shape)
print('Test set    :', X_test.shape)

# Feature scaling — important for distance-based models (KNN, SVM)
scaler = StandardScaler()
X_train_s = scaler.fit_transform(X_train)   # fit on train, transform train
X_test_s  = scaler.transform(X_test)        # transform test using train statistics

## 4. Building and Evaluating Models

### 4.1 Logistic Regression

In [ ]:
lr = LogisticRegression(max_iter=10000, random_state=RANDOM_STATE)
lr.fit(X_train_s, y_train)

y_pred_lr = lr.predict(X_test_s)
acc_lr = accuracy_score(y_test, y_pred_lr)

print(f'Logistic Regression Accuracy: {acc_lr:.4f}  ({acc_lr*100:.2f}%)')
print()
print(classification_report(y_test, y_pred_lr, target_names=['Benign', 'Malignant']))

ConfusionMatrixDisplay(
    confusion_matrix(y_test, y_pred_lr),
    display_labels=['Benign', 'Malignant']
).plot(cmap='Blues')
plt.title('Logistic Regression — Confusion Matrix')
plt.show()

### 4.2 K-Nearest Neighbours

In [ ]:
knn = KNeighborsClassifier(n_neighbors=5)
knn.fit(X_train_s, y_train)

y_pred_knn = knn.predict(X_test_s)
acc_knn = accuracy_score(y_test, y_pred_knn)

print(f'KNN Accuracy: {acc_knn:.4f}  ({acc_knn*100:.2f}%)')
print()
print(classification_report(y_test, y_pred_knn, target_names=['Benign', 'Malignant']))

ConfusionMatrixDisplay(
    confusion_matrix(y_test, y_pred_knn),
    display_labels=['Benign', 'Malignant']
).plot(cmap='Oranges')
plt.title('KNN — Confusion Matrix')
plt.show()

### 4.3 Random Forest

In [ ]:
rf = RandomForestClassifier(n_estimators=100, random_state=RANDOM_STATE)
rf.fit(X_train_s, y_train)

y_pred_rf = rf.predict(X_test_s)
acc_rf = accuracy_score(y_test, y_pred_rf)

print(f'Random Forest Accuracy: {acc_rf:.4f}  ({acc_rf*100:.2f}%)')
print()
print(classification_report(y_test, y_pred_rf, target_names=['Benign', 'Malignant']))

ConfusionMatrixDisplay(
    confusion_matrix(y_test, y_pred_rf),
    display_labels=['Benign', 'Malignant']
).plot(cmap='Greens')
plt.title('Random Forest — Confusion Matrix')
plt.show()

In [ ]:
# Bonus: top 10 feature importances from Random Forest
importances = pd.Series(rf.feature_importances_, index=X.columns)
top10 = importances.nlargest(10)

fig, ax = plt.subplots(figsize=(8, 5))
top10.sort_values().plot(kind='barh', ax=ax, color='seagreen')
ax.set_title('Random Forest — Top 10 Feature Importances')
ax.set_xlabel('Importance')
plt.tight_layout()
plt.show()

### 4.4 Support Vector Machine (SVM)

In [ ]:
svm = SVC(kernel='rbf', C=1.0, random_state=RANDOM_STATE)
svm.fit(X_train_s, y_train)

y_pred_svm = svm.predict(X_test_s)
acc_svm = accuracy_score(y_test, y_pred_svm)

print(f'SVM Accuracy: {acc_svm:.4f}  ({acc_svm*100:.2f}%)')
print()
print(classification_report(y_test, y_pred_svm, target_names=['Benign', 'Malignant']))

ConfusionMatrixDisplay(
    confusion_matrix(y_test, y_pred_svm),
    display_labels=['Benign', 'Malignant']
).plot(cmap='Purples')
plt.title('SVM — Confusion Matrix')
plt.show()

## 5. Model Comparison

In [ ]:
results = {
    'Logistic Regression': acc_lr,
    'KNN'                : acc_knn,
    'Random Forest'      : acc_rf,
    'SVM'                : acc_svm,
}

results_df = pd.DataFrame.from_dict(
    results, orient='index', columns=['Accuracy']
).sort_values('Accuracy', ascending=False)

results_df['Accuracy (%)'] = (results_df['Accuracy'] * 100).round(2)
print(results_df.to_string())
results_df

In [ ]:
fig, ax = plt.subplots(figsize=(8, 5))
colors = ['#2ecc71', '#3498db', '#e74c3c', '#9b59b6']
bars = ax.bar(results_df.index, results_df['Accuracy'], color=colors, edgecolor='white', width=0.5)

for bar, val in zip(bars, results_df['Accuracy']):
    ax.text(bar.get_x() + bar.get_width() / 2,
            bar.get_height() + 0.003,
            f'{val*100:.2f}%',
            ha='center', va='bottom', fontsize=11, fontweight='bold')

ax.set_ylim(0.90, 1.00)
ax.set_ylabel('Test Accuracy')
ax.set_title('Model Accuracy Comparison — Breast Cancer Classification')
plt.xticks(rotation=10)
plt.tight_layout()
plt.show()

## 6. Conclusion

| Model | Accuracy |
|---|---|
| Logistic Regression | 96.49% |
| KNN | 95.61% |
| **Random Forest** | **97.37%** |
| **SVM** | **97.37%** |

**Best models: Random Forest and SVM** — both achieved **97.37%** accuracy on the test set.

- **SVM** excels because the RBF kernel finds an optimal separating hyperplane in high-dimensional feature space, making it very effective when features are well-scaled.
- **Random Forest** similarly performs well by combining many decision trees, reducing variance and being robust to noisy features. Its built-in feature importance also gives interpretability.
- **Logistic Regression** is close behind (96.49%) and remains a strong, interpretable baseline.
- **KNN** is the weakest at 95.61% — it can struggle when features vary in scale or when the dataset has many dimensions (though scaling helps a lot here).

For a medical diagnosis task, **recall on the Malignant class** is especially important (minimising false negatives), so examining the full classification report alongside accuracy is critical in practice.
